In [1]:
!uv sync

Resolved 17 packages in 22ms
Audited 16 packages in 4ms


In [ ]:
# Imports
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from mandrop.engine import make_step, compute_macros, init_state
from mandrop.generator import setup, boundary_stats
from mandrop.run import run
from mandrop.viz import plot_fields

print(f"JAX {jax.__version__}, devices: {jax.devices()}")

In [ ]:
# Simulation parameters
# Lattice resolution: 1 lu = RESOLUTION_UM µm. Domain auto-sized from cropped.dxf.
#   2.5 µm/lu → 123×357 lu (fast); 1.0 µm/lu → 308×892 lu (finer).
RESOLUTION_UM       = 2.5
OUTLET_EXTRA_MM     = 0.3575    # extend outlet 2×
N_SEED_DROPLETS     = 3
DROPLET_DIAMETER_MM = 0.120

# Fluids: water + HFE-7500 + 2% PicoSurf (σ ≈ 5 mN/m, µ_HFE ≈ 1.24 cP)
# Channel depth assumed 100 µm.
W = 3.0
sigma = 0.025           # 2.5× over original; 0.05 caused NaN at 1µm/lu (κ=0.9 too stiff)
beta  = 3.0 * sigma / W
kappa = 6.0 * sigma * W

rho0 = 1.0
nu   = 1.0 / 6.0
tau_f = 3.0 * nu + 0.5
M_ch = 0.01

# Hybrid BCs: velocity on water inlets (no backflow), pressure on oil + outlet.
# Oil pressure must overcome chip back-pressure ⇒ large F_OIL.
# Water inflows kept small so the oil can actually pinch the water column.
drho                = 0.001
F_OIL               = 40.0      # rho_in_oil = rho0 + F_OIL * drho/2 = 1.020
F_OUT               = -1.0
U_TOP_IN_LU         = 0.001     # central water inflow (lu/ts)
U_WATER_SIDE_IN_LU  = 0.002     # upper-slot water inflow (lu/ts, each side)

In [ ]:
# Geometry (rasterized from cropped.dxf) and step function
geo = setup(
    resolution_um      =RESOLUTION_UM,
    outlet_extra_mm    =OUTLET_EXTRA_MM,
    n_seed_droplets    =N_SEED_DROPLETS,
    droplet_diameter_mm=DROPLET_DIAMETER_MM,
    u_top_in_lu       =U_TOP_IN_LU,
    u_water_side_in_lu=U_WATER_SIDE_IN_LU,
    rho_in_oil        =rho0 + F_OIL * drho / 2.0,
    rho_out           =rho0 + F_OUT * drho / 2.0,
)
p = geo["params"]
Nx, Ny = p["Nx"], p["Ny"]

step = make_step(
    geo["wall"], geo["fluid"], geo["interior"], geo["opp_jnp"],
    tau_f, beta, kappa, M_ch,
    geo["apply_f_bcs"], geo["apply_phi_bcs"], geo["boundary_mask"],
)

print(f"Resolution: {p['resolution_um']} µm/lu  Domain: {Nx}×{Ny}")
print(f"Channel x∈[{p['gxL']},{p['gxR']}]  Throat x∈[{p['gxTL']},{p['gxTR']}]")
print(f"Water  BC: top u_y=-{p['u_top_in_lu']:.4f}  side |u_x|={p['u_water_side_in_lu']:.4f}  (lu/ts)")
print(f"Oil/outlet rho: {p['rho_in_oil']:.4f} / {p['rho_out']:.4f}")

In [ ]:
# Initial state: equilibrium f + water prefill in upper chip region
f0, phi0 = init_state(Nx, Ny, rho0, geo["apply_phi_bcs"], geo["water_prefill"])
interior = geo["interior"]

print(f"Water prefill: {int(geo['water_prefill'].sum())} pixels")
print(f"Initial water (φ<0.5): {int((phi0 < 0.5).sum())} pixels")

In [ ]:
# Visualize geometry and initial phi distribution
fig, axes = plt.subplots(1, 2, figsize=(8, 12))

im0 = axes[0].imshow(geo["wall"].T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
axes[0].set_title('Wall mask')
axes[0].axhline(p['Y_USLOT_TOP'], color='b', ls=':', alpha=0.5, label='upper slots (water)')
axes[0].axhline(p['Y_USLOT_BOT'], color='b', ls=':', alpha=0.5)
axes[0].axhline(p['Y_LSLOT_TOP'], color='r', ls=':', alpha=0.5, label='lower slots (oil)')
axes[0].axhline(p['Y_LSLOT_BOT'], color='r', ls=':', alpha=0.5)
axes[0].legend(loc='upper right', fontsize=8)
plt.colorbar(im0, ax=axes[0], shrink=0.5)

im1 = axes[1].imshow(phi0.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
axes[1].set_title('Initial φ (oil=1, water=0)')
plt.colorbar(im1, ax=axes[1], shrink=0.5)

for ax in axes:
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Run simulation
f_final, phi_final, total_steps = run(
    step, f0, phi0, interior, p,
    chunk_size=500, n_chunks=60,
)

In [ ]:
# Summary statistics
rho_final, ux_final, uy_final = compute_macros(f_final)
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)
n_water = ((phi_final < 0.5).astype(jnp.float64) * interior).sum()

print(f"FLOW-FOCUSING — {total_steps} STEPS")
print(f"Max velocity: {float(vel_mag.max()):.4e} lu")
print(f"Water pixels: {float(n_water):.0f}")
print(f"phi range:    [{float(phi_final.min()):.6f}, {float(phi_final.max()):.6f}]")
print(f"rho range:    [{float(rho_final.min()):.6f}, {float(rho_final.max()):.6f}]")

In [ ]:
# Final field plots: φ, ρ, u_y, |u|
plot_fields(f_final, phi_final)